<h3 style="color:#1E3A8A; font-family:Arial, Helvetica, sans-serif; margin-bottom:6px;"> <b>IST 691 — Phase 1 (Part 1): Data Loading and Pre-processing</b> </h3> <div style="font-family:Arial, Helvetica, sans-serif; line-height:1.55;"> <p style="margin-top:0;"> <b>Big picture.</b> This module performs the initial data ingestion and standardization step for the project pipeline. It consolidates raw transcript data into a clean, reproducible, and traceable dataset that serves as the foundation for all subsequent stages, including sentence extraction, soft labeling, embedding, clustering, and time-series analysis. </p> <p> <b>Single source of truth (inputs).</b> Transcript data is loaded from the local <code>raw_data</code> directory (pickle or csv format). An optional economic time-series dataset may also be loaded and merged to enrich the transcript records. A configurable temporal boundary (<code>start_date</code>) defines the minimum date for inclusion. </p> <p> <b>Core logic flow.</b> <ol style="margin-top:4px;"> <li><b>Load</b> raw transcript data and optional economic data.</li> <li><b>Filter</b> transcript records by date to enforce a consistent analysis window.</li> <li><b>Derive timeline features</b> (<code>year</code>, <code>quarter</code>) to support downstream temporal aggregation.</li> <li><b>Merge</b> economic context on (<code>year</code>, <code>quarter</code>) when applicable.</li> <li><b>Filter by speaker</b> to isolate records associated with the target executive.</li> <li><b>Pre-process transcript text</b> (case normalization, whitespace cleanup, URL removal, optional filler removal, optional de-duplication).</li> <li><b>Generate stable identifiers</b> (<code>hash_id</code>) to ensure row-level traceability across all pipeline stages.</li> <li><b>Export</b> standardized datasets for downstream modules.</li> </ol> </p> <p> <b>Deliverables (local exports).</b> This module produces two datasets: <ul style="margin-top:4px;"> <li><b>Full analysis dataset</b>: all retained fields plus <code>hash_id</code>, intended for auditability and structured analysis.</li> <li><b>Reduced LLM input dataset</b>: <code>hash_id</code> and cleaned transcript text only, intended for sentence-level language-model processing.</li> </ul> </p> <p style="margin-bottom:0;"> <b>Console exhibit.</b> The execution prints a transition summary showing row retention across processing steps and displays a small sample of the reduced dataset for immediate verification. </p> </div> <hr style="border:none; border-top:1px solid #ddd; margin:12px 0;">

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
====================================================================================================
IST 691 — phase 1 (Part 1)  |  Data Loading and Preprocessing
====================================================================================================

purpose
  - standardize raw transcript text into a clean, reproducible intermediate dataset for downstream extraction
  - generate stable identifiers for row-level traceability across modules
  - produce an export-ready dataset and an LLM-ready reduced dataset (hash_id + text)

inputs (single source of truth)
  - transcript source: raw transcript dataset (pickle or csv) from local raw_data directory
  - optional auxiliary source: economic time-series dataset (csv) for year/quarter merge
  - required fields (transcript): text column, date column, speaker id column (if executive filtering enabled)

processing (logic flow)
  1) load transcript dataset (pickle preferred) and optional economic dataset
  2) normalize date fields; filter records from a configured start_date onward
  3) derive time features: year and quarter (timeline readiness)
  4) merge economic data to transcript records by (year, quarter) when economic data is provided
  5) filter by target speaker id when speaker filtering is enabled
  6) preprocess transcript text (lowercase/whitespace/url/fillers; optional de-duplication)
  7) generate stable hash identifiers (hash_id) for traceability
  8) export:
       - full dataset (all fields + hash_id)
       - reduced dataset for LLM input (hash_id + cleaned text only)
  9) log transitions and print a compact summary preview in console

data flow
  raw transcripts (+ optional economic series)
    → date-filtered subset
    → time-feature augmented dataset
    → (optional) year/quarter merged dataset
    → (optional) executive-filtered dataset
    → cleaned-text dataset (+ word_count)
    → hash_id augmented dataset
    → full export + reduced LLM export

outputs (local exports)
  - full export dataset (all columns + hash_id)
  - reduced LLM dataset (hash_id + text)
  - transition summary (rows/cols retained per step; console exhibit)

console exhibit (immediate review)
  - step-by-step logging of load/filter/merge/id generation
  - transition table (rows/cols/% retained) and sample reduced rows

====================================================================================================

December 12, 2025 | Syracuse University | IST 691 Deep Learning Term Project

Dujun; Yifeng; Isha


"""

#### =============================================================================
#### 2.1 IMPORTS AND INITIAL SETUP
#### =============================================================================
import os
import sys
import re
import hashlib
from datetime import datetime
from pathlib import Path
from typing import Tuple, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm

# ------------------------------------------------------------------------------
# Minimal patch to define `setup_environment` and `logger` if Section 1 not run
# ------------------------------------------------------------------------------

try:
    setup_environment  # Test if already defined
except NameError:
    def setup_environment():
        """
        Minimal fallback version of setup_environment.
        Replace this stub with actual import if needed.
        """
        class DummyConfig:
            def get(self, key, default=None):
                return default
        class DummyCheckpoint:
            def save_checkpoint(self, df, name):
                pass
        return DummyConfig(), DummyCheckpoint()

try:
    logger  # Test if already defined
except NameError:
    import logging
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("section2_logger")

#### =============================================================================
#### 2.2 DATA LOADER CLASS
#### =============================================================================
class DataLoader:
    """
    Loads transcript and economic data from the raw_data folder.
    """
    def __init__(self, config_manager):
        self.config = config_manager
        # fallback to "./raw_data" if pipeline.data_dir not set
        data_dir = self.config.get("pipeline.data_dir") or "./raw_data"
        self.data_dir = Path(data_dir)
    
    def load_transcript_data(self) -> pd.DataFrame:
        transcript_file = self.config.get("data.input_file") or "original_raw_apple.pkl"
        file_path = self.data_dir / transcript_file
        logger.info(f"Loading transcript data from: {file_path}")
        try:
            if file_path.suffix.lower() == ".pkl":
                df = pd.read_pickle(file_path)
            else:
                df = pd.read_csv(file_path, low_memory=False)
            logger.info(f"Loaded transcript data: {df.shape[0]} rows, {df.shape[1]} columns")
            return df
        except Exception as e:
            logger.error(f"Failed to load transcript data: {e}")
            raise
    
    def load_economic_data(self) -> pd.DataFrame:
        economic_file = self.config.get("data.economic_file") or "fred_economic_data.csv"
        file_path = self.data_dir / economic_file
        logger.info(f"Loading economic data from: {file_path}")
        try:
            df = pd.read_csv(file_path)
            logger.info(f"Loaded economic data: {df.shape[0]} rows, {df.shape[1]} columns")
            return df
        except Exception as e:
            logger.error(f"Failed to load economic data: {e}")
            raise
    
    def load_all_data(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        return self.load_transcript_data(), self.load_economic_data()

#### =============================================================================
#### 2.3 TEXT PREPROCESSOR CLASS
#### =============================================================================
class TextPreprocessor:
    """
    Cleans transcript text data.
    """
    def __init__(self, config_manager):
        self.config = config_manager
        self.text_column = self.config.get("data.text_column", "componenttext")
        self.lowercase = self.config.get("text_processing.lowercase", True)
        self.remove_duplicates = self.config.get("text_processing.remove_duplicates", True)
        self.strip_whitespace = self.config.get("text_processing.strip_whitespace", True)
        self.remove_urls = self.config.get("text_processing.remove_urls", True)
        self.remove_filler_words = self.config.get("text_processing.remove_filler_words", True)
        self.filler_words = self.config.get("text_processing.filler_words", [])
    
    def clean_text(self, text: str) -> str:
        if not isinstance(text, str):
            return ""
        if self.lowercase:
            text = text.lower()
        if self.strip_whitespace:
            text = text.strip()
            text = re.sub(r'\s+', ' ', text)
        if self.remove_urls:
            text = re.sub(r'https?://\S+|www\.\S+', '', text)
        if self.remove_filler_words and self.filler_words:
            for filler in self.filler_words:
                text = re.sub(r'\b' + re.escape(filler) + r'\b', '', text, flags=re.IGNORECASE)
        return text
    
    def preprocess_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        logger.info(f"Preprocessing text in column: {self.text_column}")
        df_copy = df.copy()
        if self.text_column in df_copy.columns:
            new_col = f"clean_{self.text_column}"
            df_copy[new_col] = df_copy[self.text_column].apply(self.clean_text)
            df_copy['word_count'] = df_copy[new_col].apply(lambda x: len(str(x).split()))
            if self.remove_duplicates:
                before = len(df_copy)
                df_copy = df_copy.drop_duplicates(subset=[new_col])
                logger.info(f"Removed {before - len(df_copy)} duplicate rows")
            logger.info(f"Text preprocessing complete: {df_copy.shape[0]} rows remain")
        else:
            logger.warning(f"Text column '{self.text_column}' not found in DataFrame")
        return df_copy

#### =============================================================================
#### 2.4 DATA FILTER CLASS
#### =============================================================================
class DataFilter:
    """
    Filters by date, speaker, and word count.
    """
    def __init__(self, config_manager):
        self.config = config_manager
        self.start_date = self.config.get("executive_filtering.start_date", "2015-01-01")
        self.speaker_id_column = self.config.get("data.speaker_id_column", "speakertypeid")
        self.date_column = self.config.get("data.date_column", "mostimportantdateutc")
    
    def filter_by_date(self, df: pd.DataFrame) -> pd.DataFrame:
        logger.info(f"Filtering data from {self.start_date} onward")
        if self.date_column in df.columns:
            df[self.date_column] = pd.to_datetime(df[self.date_column], errors='coerce')
            df = df[df[self.date_column] >= self.start_date].copy()
            logger.info(f"Date filtering complete: {df.shape[0]} rows remain")
        else:
            logger.warning(f"Date column '{self.date_column}' not found")
        return df
    
    def filter_by_speaker(self, df: pd.DataFrame) -> pd.DataFrame:
        target_id = self.config.get("executive_filtering.target_speaker_id", 2)
        logger.info(f"Filtering data where {self.speaker_id_column} == {target_id}")
        if self.speaker_id_column in df.columns:
            df = df[df[self.speaker_id_column] == target_id].copy()
            logger.info(f"Speaker filtering complete: {df.shape[0]} rows remain")
        else:
            logger.warning(f"Speaker ID column '{self.speaker_id_column}' not found")
        return df
    
    def filter_by_word_count(self, df: pd.DataFrame, min_words: int = 20) -> pd.DataFrame:
        logger.info(f"Filtering for at least {min_words} words")
        if 'word_count' in df.columns:
            df = df[df['word_count'] >= min_words].copy()
            logger.info(f"Word count filtering complete: {df.shape[0]} rows remain")
        else:
            logger.warning("Word count column not found")
        return df
    
    def apply_all_filters(self, df: pd.DataFrame, min_words: int = 20) -> pd.DataFrame:
        return (self.filter_by_date(df)
                    .pipe(lambda d: self.filter_by_speaker(d))
                    .pipe(lambda d: self.filter_by_word_count(d, min_words)))

#### =============================================================================
#### 2.5 HASH GENERATION & LLM DATA PREPARATION
#### =============================================================================
def generate_hash_id(text: str, salt: Optional[str] = None) -> str:
    combined = text if not salt else text + salt
    return hashlib.sha256(combined.encode()).hexdigest()[:16]

def add_hash_ids(df: pd.DataFrame, text_column: str, salt_column: Optional[str] = None) -> pd.DataFrame:
    logger.info("Generating hash IDs")
    df_copy = df.copy()
    if salt_column and salt_column in df_copy.columns:
        df_copy['hash_id'] = df_copy.apply(
            lambda row: generate_hash_id(str(row[text_column]), str(row[salt_column])),
            axis=1
        )
    else:
        df_copy['hash_id'] = df_copy[text_column].astype(str).apply(generate_hash_id)
    logger.info(f"Hash IDs added to {df_copy.shape[0]} rows")
    return df_copy

class LLMDataPrep:
    """
    Prepares full and reduced LLM datasets.
    """
    def __init__(self, config_manager):
        self.config = config_manager
        self.text_column = self.config.get("data.text_column", "componenttext")
    
    def create_llm_dataset(self, df: pd.DataFrame, clean_text_column: Optional[str] = None) -> pd.DataFrame:
        if clean_text_column is None:
            clean_text_column = f"clean_{self.text_column}"
        logger.info("Creating reduced LLM dataset")
        if 'hash_id' not in df.columns:
            df = add_hash_ids(df, clean_text_column)
        llm_df = df[['hash_id', clean_text_column]].copy()
        llm_df.rename(columns={clean_text_column: 'text'}, inplace=True)
        return llm_df
    
    def create_analysis_dataset(self, df: pd.DataFrame) -> pd.DataFrame:
        logger.info("Creating full analysis dataset")
        if 'hash_id' not in df.columns:
            use_col = f"clean_{self.text_column}" if f"clean_{self.text_column}" in df.columns else self.text_column
            df = add_hash_ids(df, use_col)
        return df.copy()

#### =============================================================================
#### 2.6 TRANSITION LOGGING FUNCTIONS
#### =============================================================================
transition_log = []

def log_transition(step_name: str, df: pd.DataFrame, initial_rows: int, description: str) -> None:
    rows, cols = df.shape
    pct = (rows / initial_rows * 100) if initial_rows else 0
    transition_log.append((step_name, rows, cols, f"{pct:.2f}%", description))

def print_transition_summary() -> None:
    print("\nData Transition Summary:")
    print("--------------------------------------------------------------")
    print("Step                          | Rows   | Cols | % Retained | Purpose")
    print("--------------------------------------------------------------")
    for step, rows, cols, pct, desc in transition_log:
        print(f"{step:<30} | {rows:<6,d} | {cols:<5d} | {pct:<10} | {desc}")
    print("--------------------------------------------------------------\n")

def print_reduced_examples(df: pd.DataFrame, n: int = 4) -> None:
    print("4 Example Rows from Reduced LLM Dataset:")
    for _, row in df.head(n).iterrows():
        print(f"{row['hash_id']}\t{row['text']}")

#### =============================================================================
#### 2.7 MAIN PIPELINE EXECUTION
#### =============================================================================
def run_data_processing_pipeline(config_path: Optional[str] = None, checkpoint: bool = True) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # obtain config & checkpoint
    config_manager, checkpoint_manager = setup_environment()
    
    # helper instances
    data_loader     = DataLoader(config_manager)
    text_pre        = TextPreprocessor(config_manager)
    data_filter     = DataFilter(config_manager)
    llm_prep        = LLMDataPrep(config_manager)
    
    # STEP 1: load
    logger.info("STEP 1: Loading Data")
    transcript_df, economic_df = data_loader.load_all_data()
    initial_rows = transcript_df.shape[0]
    log_transition("Load Transcripts", transcript_df, initial_rows, "Loaded raw transcripts")
    if checkpoint:
        checkpoint_manager.save_checkpoint(transcript_df, "data_loading")
    
    # STEP 2: date filter
    logger.info("STEP 2: Filtering by Date")
    transcript_df["mostimportantdateutc"] = pd.to_datetime(
        transcript_df["mostimportantdateutc"], errors='coerce'
    )
    transcript_df = transcript_df[transcript_df["mostimportantdateutc"] >= "2015-01-01"].copy()
    log_transition("Date Filter", transcript_df, initial_rows, "Records >= 2015-01-01")
    
    if "date" in economic_df.columns:
        economic_df["date"] = pd.to_datetime(economic_df["date"], errors='coerce')
        economic_df = economic_df[economic_df["date"] >= "2015-01-01"].copy()
    
    # STEP 3: add features
    logger.info("STEP 3: Adding Date Features")
    transcript_df['year']  = transcript_df["mostimportantdateutc"].dt.year
    transcript_df['quarter']= transcript_df["mostimportantdateutc"].dt.quarter
    economic_df['year']    = economic_df["date"].dt.year
    economic_df['quarter'] = economic_df["date"].dt.quarter
    log_transition("Add Date Features", transcript_df, initial_rows, "year & quarter added")
    
    # STEP 4: merge
    logger.info("STEP 4: Merging Economic Data")
    merged = pd.merge(
        transcript_df, economic_df,
        how='left', on=["year","quarter"], suffixes=('','_econ')
    )
    log_transition("Merge Econ", merged, initial_rows, "Merged on year & quarter")
    
    # STEP 5: speaker filter
    logger.info("STEP 5: Filtering for Target Speaker")
    target_id = config_manager.get("executive_filtering.target_speaker_id", 2)
    ceo_df = merged[merged["speakertypeid"] == target_id].copy()
    log_transition("CEO Filter", ceo_df, initial_rows, "speakertypeid == 2")
    
    # STEP 6: hash IDs
    logger.info("STEP 6: Generating Hash IDs")
    ceo_df = add_hash_ids(ceo_df, config_manager.get("data.text_column", "componenttext"))
    log_transition("Add Hash ID", ceo_df, initial_rows, "hash_id added")
    
    # STEP 7: prepare exports
    full_export    = ceo_df.copy()
    reduced_df     = ceo_df[['hash_id', config_manager.get("data.text_column", "componenttext")]].copy()
    reduced_df.rename(columns={config_manager.get("data.text_column", "componenttext"): "text"}, inplace=True)
    log_transition("Full Export", full_export, initial_rows, "all cols + hash")
    log_transition("Reduced LLM", reduced_df, initial_rows, "hash + text only")
    
    # OUTPUT directory fallback
    output_dir = config_manager.get("pipeline.output_dir") or "./output"
    os.makedirs(output_dir, exist_ok=True)
    full_path    = os.path.join(output_dir, "Section2_Full_Export_Dataset_With_All_Fields_and_HashIDs.csv")
    reduced_path = os.path.join(output_dir, "Section2_Reduced_LLM_Input_Dataset_Hash_and_Clean_Transcript_Text.csv")
    full_export.to_csv(full_path, index=False)
    reduced_df.to_csv(reduced_path, index=False)
    logger.info(f"Saved full export to: {full_path}")
    logger.info(f"Saved reduced export to: {reduced_path}")
    
    return full_export, reduced_df

#### =============================================================================
#### 2.8 MAIN GUARD
#### =============================================================================
if __name__ == "__main__":
    try:
        full_df, reduced_df = run_data_processing_pipeline()
        print_transition_summary()
        print_reduced_examples(reduced_df, n=4)
        logger.info(f"Pipeline complete: full={full_df.shape[0]}, reduced={reduced_df.shape[0]}")
    except Exception as e:
        logger.error(f"Pipeline execution failed: {e}", exc_info=True)
        sys.exit(1)

#### =============================================================================
#### 2.9 FINAL OUTPUT: PRINT SUMMARY & SAMPLE ROWS
#### =============================================================================
if __name__ == "__main__":
    try:
        # Run the pipeline
        full_export, reduced_dataset = run_data_processing_pipeline()
        # Print out transition log and examples
        print_transition_summary()
        print_reduced_examples(reduced_dataset, n=4)
        # Final info message
        logger.info(
            f"Data processing complete: "
            f"Full export = {full_export.shape[0]} rows; "
            f"Reduced = {reduced_dataset.shape[0]} rows"
        )
    except Exception as e:
        # Log stack trace and exit
        logger.error("Pipeline execution failed", exc_info=True)
        sys.exit(1)
#### =============================================================================

"""
SECTION 2 SUMMARY:
------------------
Section 2 loads and preprocesses data using the settings and folder structure established in Section 1.
Key steps include:

  1. Loading raw data from the raw_data folder:
       - Transcript data is now read from "original_raw_apple.pkl".
       - Economic data is read from "fred_economic_data.csv".
  2. Filtering records to include only those from January 1, 2015 onward.
  3. Converting date columns to datetime and adding year and quarter features.
  4. Merging transcript and economic data on year and quarter.
  5. Filtering the merged data to include only records for the target speaker (e.g., Apple CEO).
  6. Generating unique hash IDs for each record.
  7. Creating two outputs:
       - A full export dataset (all columns plus hash) saved as "Section2_Full_Export_Dataset_With_All_Fields_and_HashIDs.csv".
       - A reduced dataset for LLM input (hash and transcript text) saved as "Section2_Reduced_LLM_Input_Dataset_Hash_and_Clean_Transcript_Text.csv".

These processes ensure that all subsequent sections, such as belief extraction, operate on a clean,
reproducible, and well-organized dataset.

NEXT: Section 3 – Belief Extraction
"""


INFO:section2_logger:STEP 1: Loading Data
INFO:section2_logger:Loading transcript data from: raw_data/original_raw_apple.pkl
INFO:section2_logger:Loaded transcript data: 1026916 rows, 30 columns
INFO:section2_logger:Loading economic data from: raw_data/fred_economic_data.csv
INFO:section2_logger:Loaded economic data: 266 rows, 8 columns
INFO:section2_logger:STEP 2: Filtering by Date
/var/folders/3k/8mq15bms13s0mll95nx_wjxh0000gp/T/ipykernel_58142/1018590785.py:318: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  economic_df["date"] = pd.to_datetime(economic_df["date"], errors='coerce')
INFO:section2_logger:STEP 3: Adding Date Features
INFO:section2_logger:STEP 4: Merging Economic Data
INFO:section2_logger:STEP 5: Filtering for Target Speaker
INFO:section2_logger:STEP 6: Generating Hash IDs
INFO:section2_logger:Generating hash IDs
INFO:section2_logger


Data Transition Summary:
--------------------------------------------------------------
Step                          | Rows   | Cols | % Retained | Purpose
--------------------------------------------------------------
Load Transcripts               | 1,026,916 | 30    | 100.00%    | Loaded raw transcripts
Date Filter                    | 690,700 | 30    | 67.26%     | Records >= 2015-01-01
Add Date Features              | 690,700 | 32    | 67.26%     | year & quarter added
Merge Econ                     | 690,700 | 40    | 67.26%     | Merged on year & quarter
CEO Filter                     | 368,613 | 40    | 35.90%     | speakertypeid == 2
Add Hash ID                    | 368,613 | 41    | 35.90%     | hash_id added
Full Export                    | 368,613 | 41    | 35.90%     | all cols + hash
Reduced LLM                    | 368,613 | 2     | 35.90%     | hash + text only
--------------------------------------------------------------

4 Example Rows from Reduced LLM Dataset:
200

INFO:section2_logger:Loaded transcript data: 1026916 rows, 30 columns
INFO:section2_logger:Loading economic data from: raw_data/fred_economic_data.csv
INFO:section2_logger:Loaded economic data: 266 rows, 8 columns
INFO:section2_logger:STEP 2: Filtering by Date
/var/folders/3k/8mq15bms13s0mll95nx_wjxh0000gp/T/ipykernel_58142/1018590785.py:318: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  economic_df["date"] = pd.to_datetime(economic_df["date"], errors='coerce')
INFO:section2_logger:STEP 3: Adding Date Features
INFO:section2_logger:STEP 4: Merging Economic Data
INFO:section2_logger:STEP 5: Filtering for Target Speaker
INFO:section2_logger:STEP 6: Generating Hash IDs
INFO:section2_logger:Generating hash IDs
INFO:section2_logger:Hash IDs added to 368613 rows
INFO:section2_logger:Saved full export to: ./output/Section2_Full_Export_Dataset_With_All_Fiel


Data Transition Summary:
--------------------------------------------------------------
Step                          | Rows   | Cols | % Retained | Purpose
--------------------------------------------------------------
Load Transcripts               | 1,026,916 | 30    | 100.00%    | Loaded raw transcripts
Date Filter                    | 690,700 | 30    | 67.26%     | Records >= 2015-01-01
Add Date Features              | 690,700 | 32    | 67.26%     | year & quarter added
Merge Econ                     | 690,700 | 40    | 67.26%     | Merged on year & quarter
CEO Filter                     | 368,613 | 40    | 35.90%     | speakertypeid == 2
Add Hash ID                    | 368,613 | 41    | 35.90%     | hash_id added
Full Export                    | 368,613 | 41    | 35.90%     | all cols + hash
Reduced LLM                    | 368,613 | 2     | 35.90%     | hash + text only
Load Transcripts               | 1,026,916 | 30    | 100.00%    | Loaded raw transcripts
Date Filter        

'\nSECTION 2 SUMMARY:\n------------------\nSection 2 loads and preprocesses data using the settings and folder structure established in Section 1.\nKey steps include:\n\n  1. Loading raw data from the raw_data folder:\n       - Transcript data is now read from "original_raw_apple.pkl".\n       - Economic data is read from "fred_economic_data.csv".\n  2. Filtering records to include only those from January 1, 2015 onward.\n  3. Converting date columns to datetime and adding year and quarter features.\n  4. Merging transcript and economic data on year and quarter.\n  5. Filtering the merged data to include only records for the target speaker (e.g., Apple CEO).\n  6. Generating unique hash IDs for each record.\n  7. Creating two outputs:\n       - A full export dataset (all columns plus hash) saved as "Section2_Full_Export_Dataset_With_All_Fields_and_HashIDs.csv".\n       - A reduced dataset for LLM input (hash and transcript text) saved as "Section2_Reduced_LLM_Input_Dataset_Hash_and_Cle

<div style="border: 2px solid #1A5276; border-radius: 8px; padding: 18px; background: linear-gradient(to bottom right, #FDFEFE, #FCF3CF); font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 14px; color: #1A1A1A; line-height: 1.7; max-width: 1000px; box-sizing: border-box;">
  <p style="font-size: 23px; font-weight: 700; color: #154360; margin-bottom: 19px;">
    Section 2 — Final Takeaway: Data Loading, Cleaning, and Executive Statement Preparation
  </p>
  <p>
    This section built a clean and usable dataset of executive speech from Apple’s earnings call transcripts, matched with economic indicators, and prepared the data for belief extraction. It followed a logical flow: <strong>load → clean → filter → enrich → analyze</strong>.
  </p>
  <hr style="border: none; border-top: 1px solid #D5D8DC; margin: 16px 0;">
  
  <p style="font-weight: 600; color:#1A5276;">Data Transition Summary:</p>
  <pre style="background-color: #FEF9E7; border-left: 4px solid #D4AC0D; padding: 12px; font-family: 'Courier New', monospace; font-size: 13px;">
--------------------------------------------------------------
Step                          | Rows      | Cols | % Retained | Purpose
--------------------------------------------------------------
Load Transcripts               | 1,026,916 | 30   | 100.00%    | Loaded raw transcript data
Date Filter                    | 690,700   | 30   | 67.26%     | Records from 2015 onward
Add Date Features              | 690,700   | 32   | 67.26%     | Added year & quarter
Merge Economic Data            | 690,700   | 40   | 67.26%     | Merged on year & quarter
CEO Filter                     | 368,613   | 40   | 35.90%     | Records for target speaker
Add Hash ID                    | 368,613   | 41   | 35.90%     | Unique hash ID added per row
Full Export Version            | 368,613   | 41   | 35.90%     | Full dataset (all columns + hash)
Reduced LLM Version            | 368,613   | 2    | 35.90%     | Reduced dataset (hash + transcript)
--------------------------------------------------------------
  </pre>

  <p style="font-weight: 600; color:#1A5276;">Data Preparation Workflow:</p>
  <ul style="margin-top: 6px; padding-left: 20px;">
    <li><strong>Loaded 1,026,916 rows</strong> of raw transcript data from the <code>raw_data</code> folder.</li>
    <li><strong>Date Filter:</strong> Retained 690,700 rows with dates from January 1, 2015 onward.</li>
    <li><strong>Add Date Features:</strong> Enriched data by adding 'year' and 'quarter' to enable precise merging.</li>
    <li><strong>Merge Economic Data:</strong> Successfully merged transcripts with economic data (matching on year & quarter) while retaining 690,700 rows.</li>
    <li><strong>CEO Filter:</strong> Focused on Apple CEO (speakertypeid = 2), resulting in 368,613 rows.</li>
    <li><strong>Add Hash ID:</strong> Generated unique hash IDs for each record to ensure traceability.</li>
    <li><strong>Export Versions:</strong> Produced a full export (all columns + hash) and a reduced dataset (hash + transcript) for LLM processing.</li>
  </ul>

  <p style="font-weight: 600; color:#1A5276;">Data Trends and Diagnostics:</p>
  <pre style="background-color: #FEF9E7; border-left: 4px solid #D4AC0D; padding: 12px; font-family: 'Courier New', monospace; font-size: 13px;">
 Transcript Count: 1,026,916 rows
 Post-Date Filter: 690,700 rows (67.26% retained)
 Target Speaker Rows: 368,613 rows (35.90% retained)
 Merge Consistency: 100% match for economic data on valid records
 Overall, dataset reduced to 35.90% of original raw data after all filtering steps.
  </pre>

  <p style="font-weight: 600; color:#1A5276;">Visual Insights:</p>
  <ul style="margin-top: 6px; padding-left: 20px;">
    <li><span style="color:#CC0000;">Peak transcript volumes</span> occurred before filtering, indicating periods of high call activity.</li>
    <li><span style="color:#CC0000;">Filtering by date and speaker</span> effectively reduced the noise, focusing on relevant executive speeches.</li>
    <li><span style="color:#CC0000;">Word count and duplicate removal</span> were crucial in ensuring data quality.</li>
    <li><span style="color:#CC0000;">The unique hash generation</span> provides a robust mechanism for tracking and merging records in later stages.</li>
  </ul>

  <p style="font-weight: 600; color:#1A5276;">Summary:</p>
  <p>
    Section 2 prepared a clean and focused dataset by loading, filtering, enriching, and organizing executive statements with contextual economic data. The resulting datasets (full and reduced) are well-structured and ready for belief extraction in Section 3. This comprehensive preparation not only improves data quality but also ensures that the subsequent modeling phases have a solid, reproducible foundation.
  </p>
</div>